# Cue — a real-time presentation *rescue* agent

**Cue** helps a speaker recover in real time when they hit an unexpected question or slip during a
presentation, live sale, or video shoot. Upload your own documents → Cue indexes them (RAG) → when a
rescue is needed it generates a short, **grounded, ready-to-read script** from *your* material, shown
line-by-line like song lyrics.

*Capstone for the **5-Day AI Agents: Intensive Vibe Coding Course with Google**.*

- 🎥 **Demo video:** https://youtu.be/9WT2cFZ4iuA
- 💻 **Backend repo:** https://github.com/Intensive-Vibe-Coding-Capstone-Project/Backend
- 🤖 **Built with** Google Gemini (product runtime) + Python / FastAPI.

> **Team project of two.** Both members collaborated on this build; each submits a Writeup that
> references this same notebook, repo, and video.

---
This notebook is **runnable and reproducible with no API key** — it uses Cue's built-in deterministic
*fake* providers so every cell executes offline on Kaggle. An optional cell shows the live Gemini path
if you attach a `GEMINI_API_KEY` secret.

## 1. The problem & who it's for

People who present, sell live, or record content prepare heavily but still get caught off-guard by:

- **a hard / unexpected question** they didn't rehearse,
- **saying the wrong thing** (a creator naming the wrong brand on a sponsored stream), or
- **losing the thread** mid-recording.

Today they freeze, or stall while googling. **Cue removes that gap** — a few seconds of ready-to-read
script, grounded in the speaker's own material, so it sounds like *them*, not a generic chatbot.

**Users:** student presenters defending a project · office workers pitching under pressure ·
content creators / KOLs selling or recording live.

## 2. What it does — the four use cases

All four are covered end-to-end in the backend:

1. **Hard question mid-talk** — repeat the question → `POST /rescue` (or the live WS trigger) →
   grounded next lines to say.
2. **Hard question in Q&A** — same path, after the talk.
3. **Live-sale slip** — a ~30s scan detects a spoken brand diverging from the prepared script →
   an on-message correction line.
4. **Recording flow break** — off-flow speech (low overlap with the prepared script) →
   a recovery suggestion.

## 3. How it works (architecture)

```
                 ┌──────────────┐
  Upload docs ──▶│  Ingestion   │── parse → chunk → embed ─▶ ┌───────────┐
  (pdf/docx/...) └──────────────┘                            │ Chroma DB  │
                                                             └─────┬──────┘
  Transcript ──▶ WS /stream ──▶ Transcript buffer ──┐              │ retrieve
  (text MVP)                                         ▼              ▼
                                          ┌────────────────────────────┐
   Trigger (button / 30s / keyword /      │    Rescue Orchestrator     │──▶ Gemini Flash ──▶ rescue script
   silence)────────────────────────▶│  (RAG + window + script)   │            │
                                          └────────────────────────────┘            ▼
                                                    │ persist turn        line-by-line render
                                                    ▼                       (lyrics style)
                                              Session store (SQLite)
```

**Pipeline:** ingestion (parse pdf/docx/txt/pptx/epub → normalize → chunk → Gemini embeddings → Chroma)
→ retrieval (top-k with citations `doc_id:chunk_index`) → rescue orchestrator (prompt = retrieved
passages + transcript window + prepared script) → Gemini Flash → lyric-line rendering. Sessions/turns
persist in SQLite; the live path streams transcript over WebSocket and fires triggers.

**Grounding (two layers + a guard):** a retrieval score floor hard-bridges obviously-empty retrieval
*without* an LLM call; the prompt says "answer only from context, else return a safe bridge line"; and
**refusal detection** is the primary guard (a model "I don't have that" → `grounded=False`). A rescue
with no support returns a safe **bridge line**, never an invented fact.

## 4. AI agent design + how we built it

**Product runtime:** a Gemini-based **rescue orchestrator** — a RAG agent that fuses retrieved
passages, the rolling transcript window, and the prepared script, then decides between a grounded
script and a safe refusal. A **trigger engine** (manual / periodic / keyword / silence) decides *when*
it fires; a lexical **slip detector** (brand + off-flow) covers the live-sale / recording cases.
Providers sit behind protocols (`gemini` | `fake` | `auto`), so the whole system runs deterministically
offline — which is exactly why this notebook can run without a key.

**How we built it — Loop Engineering** (a course differentiator): three Claude-Code agents drive
development. *These agents build the app; they are not the app.*

```
   ┌──────────┐     plan      ┌──────────┐    code     ┌──────────┐
   │ PLANNER  │ ──────────▶ │   DEV    │ ─────────▶ │    QA    │
   └──────────┘               └──────────┘             └────┬─────┘
        ▲                                                   │ pass/fail
        │            update context (progress, decisions)   │
        └────────────────────────────────────┘
```

One trip = one roadmap task, grounded in a **Context Engineering** docs system
(`docs/` + `context/` + `.claude/skills/`).

## 5. Run it yourself (keyless & deterministic)

The cells below install Cue from GitHub and run the full **index → retrieve → ground → generate**
pipeline using the offline *fake* providers — **no API key, no network calls to Gemini**.

> ⚠️ **Kaggle setup:** turn **Internet = On** (right sidebar → Notebook options) so `pip` can install
> from GitHub. The repo must be public.

In [ ]:
# Install Cue (+ Chroma vector store) straight from the repo.
# google-genai comes with [rag] too, but is only used on the optional live path below.
!pip install -q "cue[rag] @ git+https://github.com/Intensive-Vibe-Coding-Capstone-Project/Backend.git"

In [ ]:
# Configure the OFFLINE, deterministic providers BEFORE importing cue.
# (get_settings() is cached on first use, so env must be set first.)
import os

os.environ["CUE_EMBEDDINGS_PROVIDER"] = "fake"   # hashed bag-of-words embedder
os.environ["CUE_GENERATION_PROVIDER"] = "fake"   # echoes retrieved sentences, never invents
os.environ["CUE_RESCUE_MIN_SCORE"]    = "0.3"    # fake embedder is on a different scale than Gemini
os.environ["CUE_CHROMA_DIR"]          = "/kaggle/working/chroma"
os.environ["CUE_DB_PATH"]             = "/kaggle/working/cue.db"

from cue.config import get_settings
s = get_settings()
print("embeddings:", s.active_embeddings_provider,
      "| generation:", s.active_generation_provider,
      "| min_score:", s.rescue_min_score)

In [ ]:
# Index one of the speaker's own documents (an FAQ for a fictional product, "AuroraBuds Pro").
from cue.ingestion.models import DocType, ParsedDocument
from cue.rag import index

DOC = """AuroraBuds Pro - Frequently Asked Questions

Q: Can I wear AuroraBuds Pro at the gym or running in the rain?
A: Yes. AuroraBuds Pro are IPX5 rated, so they are sweat resistant and handle light rain.
Gym workouts and running in the rain are fine. They are not designed for swimming or submersion.

Q: How long does the battery last?
A: The earbuds last 8 hours on a single charge, and the charging case adds three full charges
for about 32 hours of total listening time.

Q: What makes the sound personalized?
A: AuroraBuds Pro run a short in-app hearing calibration and tune the audio profile to your ears.
"""

meta = index.index_document(
    ParsedDocument(
        id="doc1",
        filename="aurorabuds-faq.txt",
        doc_type=DocType.txt,
        char_count=len(DOC),
        text=DOC,
    )
)
print(f"indexed '{meta.filename}': {meta.n_chunks} chunk(s), {meta.char_count} chars")

In [ ]:
# USE CASE 1 - a hard question mid-talk. Cue retrieves from the doc and returns a GROUNDED script.
from cue.rescue import service

resp = service.generate_rescue("Can I wear these at the gym or running in the rain?")

print("grounded:", resp.grounded)
print("--- rescue script (line by line, karaoke style) ---")
for line in resp.lines:
    print("  ", line)
print("--- citations (doc_id:chunk_index) ---")
for c in resp.citations:
    print(f"   {c.doc_id}:{c.chunk_index}  {c.filename}  score={c.score:.3f}")

In [ ]:
# Ask something the documents do NOT cover. Cue must NOT invent an answer -
# it returns a safe "bridge" line and grounded=False. No hallucinated facts.
resp = service.generate_rescue("What is a good recipe for chocolate cake?")

print("grounded:", resp.grounded)
print("bridge line:", resp.script)
assert resp.grounded is False, "off-topic query should not be grounded"
print("\nOK - off-topic correctly refused on the deterministic offline path.")

## 6. Results / eval (measured, honest)

**Offline gate (reproducible CI signal — what this notebook runs):**
- `pytest` **66/66** green, keyless; `ruff check` + `format --check` clean.
- 13-case grounding eval classifies grounded-vs-bridge **correctly (13/13)** on the deterministic fake
  embedder (this is why the off-topic cell above refuses reliably).

**Live eval (real Gemini, throttled to the 5 req/min free tier):**
- **Grounding: 11/13 = 85%** classification accuracy — **below the ≥90% target.**
  - On-topic: **10/10** grounded correctly, with citations.
  - Off-topic: **1/3** correctly refused; **2/3 leaked** a grounded script for nonsense.
- **Latency:** p50 **4.08s** (≈ the ~4s budget); p95/max **7.60s** (over budget).

**Known limit (stated, not hidden):** off-topic over-grounding on the *live* path. Gemini embeddings are
anisotropic — even nonsense scores above the score floor — so the LLM gets called, and refusal detection
caught only 2/3 off-topic cases that run. **Fix in progress:** strengthen the refusal prompt / add a
post-generation relevance check (without over-tightening the floor, which would start refusing real
content). Every error path still degrades to the safe bridge — no crash, no invented fact.

## 7. Reproducibility

- **Python** ≥3.11. **Install:** `pip install "cue[rag,parsers] @ git+<repo>"`.
- **Models / store (config, never hard-coded):** rescue = `gemini-2.5-flash` (temp 0.3, capped output),
  embeddings = `gemini-embedding-001`, vector store = **Chroma** (local persistent), sessions = SQLite.
- **Secrets:** `GEMINI_API_KEY` only — via **Kaggle Secrets** (or `.env` locally; no key is committed).
- **Determinism:** with **no key**, embeddings + generation fall back to deterministic **fakes** (set
  above), so this notebook and `pytest` are fully reproducible with no network.
- **Run locally:** `pytest` (keyless) · `cue` (dev server + `/ui` demo) ·
  `python tests/eval/run_eval.py` (grounding + latency; live with a key, offline without).

## 8. Optional: the live Gemini path

To run Cue against **real Gemini**, add a Kaggle Secret named `GEMINI_API_KEY`
(**Add-ons → Secrets**), enable Internet, then run the cell below. Without the secret it skips cleanly,
so *Save & Run All* still succeeds on the keyless path above.

In [ ]:
# OPTIONAL live path - safe to run; skips if no GEMINI_API_KEY secret is attached.
key = None
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    key = os.environ.get("GEMINI_API_KEY")

if not key:
    print("No GEMINI_API_KEY secret found - skipping the live path (keyless demo above is enough).")
else:
    # Re-point providers to Gemini. Drop the fake overrides -> providers resolve to "auto"
    # (= gemini when a key is present) with the default 0.4 score floor.
    for k in ("CUE_EMBEDDINGS_PROVIDER", "CUE_GENERATION_PROVIDER", "CUE_RESCUE_MIN_SCORE"):
        os.environ.pop(k, None)
    os.environ["GEMINI_API_KEY"] = key
    # Fresh store: Gemini embeddings have a different dimension than the fakes above,
    # and get_store() is cached per Chroma dir - a new path gives a clean collection.
    os.environ["CUE_CHROMA_DIR"] = "/kaggle/working/chroma_live"
    os.environ["CUE_DB_PATH"]    = "/kaggle/working/cue_live.db"
    get_settings.cache_clear()

    # index / service (imported above) read settings at call time - no reload needed.
    index.index_document(
        ParsedDocument(id="doc1", filename="aurorabuds-faq.txt",
                       doc_type=DocType.txt, char_count=len(DOC), text=DOC)
    )
    live = service.generate_rescue("Can I wear these at the gym or running in the rain?")
    print("grounded:", live.grounded)
    for line in live.lines:
        print("  ", line)

## 9. Future work

Bi-directional **live audio STT** (MVP ingests transcript text), token **streaming** to cut p95 latency,
multi-language, fine-tuned signal detection, auth / team workspaces, and deploy (Cloud Run + pgvector).
Immediate next: close the off-topic grounding gap to clear ≥90%.

---

**Cue — your real-time rescue, grounded in your own words. Never freeze on stage again.**

🎥 https://youtu.be/9WT2cFZ4iuA · 💻 https://github.com/Intensive-Vibe-Coding-Capstone-Project/Backend